# Early OP engagement within the same discussion

**Does the association persist when branches under the same original post have the same number of early non-OP comments?**

This notebook reuses the full training [branch dataset](op_engagement_study.ipynb). The first 24 hours begin when each branch opens. The outcome is the first verified OP-awarded delta in the next seven days, among branches still at risk at hour 24. We use the same eligibility and timestamp checks as the original analysis.

In [1]:
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import display


REPO = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "team-setup").is_dir() and (path / "pyproject.toml").is_file()
)
dataset_path = REPO / "project/data/processed/op-engagement-study/branches.parquet"
if not dataset_path.is_file():
    script = (
        REPO / "project/notebooks/nested-replies-analysis/op_engagement_study/build_branch_dataset.py"
    )
    subprocess.run([sys.executable, str(script)], check=True)

columns = [
    "post_id", "branch_id", "eligible", "comments_24h", "op_replies_24h", "delta_after_24h",
]
branches = pd.read_parquet(dataset_path, columns=columns)
eligible = branches.loc[branches["eligible"]].copy()

# The total counts all archived descendant comments. Subtract the OP's own turns.
for column in ("comments_24h", "op_replies_24h", "delta_after_24h"):
    eligible[column] = eligible[column].astype(int)
eligible["non_op_comments_24h"] = eligible["comments_24h"] - eligible["op_replies_24h"]
eligible["op_engaged_24h"] = eligible["op_replies_24h"].gt(0)
assert eligible["non_op_comments_24h"].ge(0).all()
display(eligible[[
    "post_id", "branch_id", "comments_24h", "op_replies_24h", "non_op_comments_24h",
    "op_engaged_24h", "delta_after_24h",
]].head())

,post_id,branch_id,comments_24h,op_replies_24h,non_op_comments_24h,op_engaged_24h,delta_after_24h
4,t3_16ra7x,t1_c8busfv,0,0,0,False,0
5,t3_16ra7x,t1_c8dtzfw,0,0,0,False,0
6,t3_16ra7x,t1_c8eim1m,0,0,0,False,0
38,t3_16t9ct,t1_c87bu7k,0,0,0,False,0
47,t3_16uat5,t1_c8bmnc9,0,0,0,False,0


## Find branches we can compare

Match exactly on the original post and the first-day non-OP comment count. Keep a group only if it contains branches both with and without an early OP reply. A count of zero can qualify because a branch can have an OP reply and zero replies by anyone else. The non-OP count includes archived bot comments, consistent with the source table's total-comment definition.

In [2]:
# One comparison group is a post paired with an exact non-OP comment count.
match_keys = ["post_id", "non_op_comments_24h"]
totals = eligible.groupby([*match_keys, "op_engaged_24h"]).agg(
    branches=("branch_id", "size"),
    later_deltas=("delta_after_24h", "sum"),
)
sizes = totals["branches"].unstack("op_engaged_24h", fill_value=0).reindex(
    columns=[False, True], fill_value=0
)
events = totals["later_deltas"].unstack("op_engaged_24h", fill_value=0).reindex(
    columns=[False, True], fill_value=0
)
overlap = sizes.min(axis=1).gt(0)
strata = pd.DataFrame({
    "no_op_branches": sizes.loc[overlap, False],
    "op_branches": sizes.loc[overlap, True],
    "no_op_later_deltas": events.loc[overlap, False],
    "op_later_deltas": events.loc[overlap, True],
})
if strata.empty:
    raise ValueError("No same-post, same-count groups include both OP categories.")

# Report how much of the cohort supports a within-discussion comparison.
coverage = pd.DataFrame({
    "Eligible training cohort": [
        eligible["post_id"].nunique(), len(sizes), len(eligible), eligible["delta_after_24h"].sum(),
    ],
    "With both OP groups": [
        strata.index.get_level_values("post_id").nunique(), len(strata),
        strata[["no_op_branches", "op_branches"]].to_numpy().sum(),
        strata[["no_op_later_deltas", "op_later_deltas"]].to_numpy().sum(),
    ],
}, index=["Discussions", "Post × exact count groups", "Branches", "Later deltas"])
display(coverage)
display(strata.groupby(level="non_op_comments_24h").sum().head(10))

,Eligible training cohort,With both OP groups
Discussions,17981,8611
Post × exact count groups,76343,11872
Branches,207564,69549
Later deltas,404,100


,no_op_branches,op_branches,no_op_later_deltas,op_later_deltas
non_op_comments_24h,,,,
0,40873,10895,52,27
1,5292,5177,3,10
2,1791,1769,0,4
3,724,731,0,1
4,403,409,1,1
5,222,234,0,1
6,157,152,0,0
7,64,65,0,0
8,62,61,0,0


## Compare later delta rates within those groups

Give both OP categories the same weight in each group. The primary estimate weights each group by the size of its smaller category, so a group with 100 branches on one side and one on the other contributes the same weight as a group with one on each side. As a sensitivity check, we also give each discussion equal weight after averaging its matched groups. We use 1,000 bootstrap resamples of entire discussions for the 95% intervals.

This comparison is descriptive. OP responses can themselves prompt more non-OP activity during the first day, and branches can open at different times under the same post.

In [3]:
strata["weight"] = strata[["no_op_branches", "op_branches"]].min(axis=1)
strata["weighted_no_op_rate"] = (
    strata["weight"] * strata["no_op_later_deltas"] / strata["no_op_branches"]
)
strata["weighted_op_rate"] = (
    strata["weight"] * strata["op_later_deltas"] / strata["op_branches"]
)

# Each bootstrap unit is a whole post. Its weight and both weighted rates travel together.
post_totals = strata[["weight", "weighted_no_op_rate", "weighted_op_rate"]].groupby(level="post_id").sum()
weighted_rates = post_totals.iloc[:, 1:].div(post_totals["weight"], axis=0)
post_estimates = np.column_stack([post_totals.to_numpy(), weighted_rates.to_numpy()])


def rates_and_differences(totals, n_posts):
    """Return the two adjusted rates and their difference for each weighting scheme."""
    overlap_weighted = totals[..., 1:3] / totals[..., :1]
    equal_discussion = totals[..., 3:5] / n_posts
    both = np.stack([overlap_weighted, equal_discussion], axis=-2)
    return np.concatenate([both, (both[..., 1] - both[..., 0])[..., None]], axis=-1)


n_posts = len(post_totals)
rng = np.random.default_rng(42)
observed_rates = rates_and_differences(post_estimates.sum(axis=0), n_posts)
bootstrap = []
for _ in range(20):
    draws = rng.multinomial(n_posts, np.full(n_posts, 1 / n_posts), size=50)
    bootstrap.append(rates_and_differences(draws @ post_estimates, n_posts))
lower, upper = np.quantile(np.concatenate(bootstrap), [0.025, 0.975], axis=0)

estimates = pd.DataFrame({
    "No early OP reply (%)": 100 * observed_rates[:, 0],
    "Early OP reply (%)": 100 * observed_rates[:, 1],
    "Difference (percentage points)": 100 * observed_rates[:, 2],
    "95% CI lower (pp)": 100 * lower[:, 2],
    "95% CI upper (pp)": 100 * upper[:, 2],
}, index=["Weighted by the smaller group", "Equal weight per discussion"])
display(estimates.round(3))

,No early OP reply (%),Early OP reply (%),Difference (percentage points),95% CI lower (pp),95% CI upper (pp)
Weighted by the smaller group,0.161,0.197,0.036,-0.051,0.126
Equal weight per discussion,0.198,0.223,0.024,-0.086,0.133
